In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

driver.get("http://localhost:5173/")
driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
driver.get("http://localhost:5173/login")

wait.until(EC.presence_of_element_located((By.ID, "username")))
driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
driver.find_element(By.ID, "password").send_keys("12345678")
driver.find_element(By.ID, "sign-in-btn").click()

time.sleep(3)
print("URL:", driver.current_url)
print("Page text:", driver.find_element(By.TAG_NAME, "body").text[:200])

In [ ]:
try:
    # Go to POS / Sales (sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'POS / Sales')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='POS / Sales']")))

    # Sensitive medicines show a "Restricted" pill in their catalog row (verified in CatalogTable.jsx)
    wait.until(EC.presence_of_element_located((By.XPATH, "//tr[contains(@class, 'pos-row')]")))
    rows = [r for r in driver.find_elements(By.XPATH, "//tr[contains(@class, 'pos-row')]") if r.is_displayed() and "Restricted" in r.text]
    assert rows, "No sensitive (Restricted) medicine found in the catalog."
    row = rows[0]
    med_name = row.text.split("\n")[0]
    print("Sensitive medicine:", med_name)
    add = [b for b in row.find_elements(By.XPATH, ".//button[contains(., 'Add')]") if b.is_displayed() and b.is_enabled()]
    assert add, "Add button is disabled for the sensitive medicine."
    add[0].click()
    time.sleep(2)
    assert med_name in driver.find_element(By.TAG_NAME, "body").text, "Medicine not added to cart."

    # Try to complete the sale (button text "Complete Sale \u00b7 ৳X", verified in PaymentCard.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Complete Sale')]"))).click()
    time.sleep(3)

    # The real warning modal must appear (verified in InteractionReviewModal.jsx)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//*[text()='Sensitive Medicine Approval']")))
    body = driver.find_element(By.TAG_NAME, "body").text
    assert "Sensitive medicine requires staff approval before completing the sale." in body
    print("Warning text: Sensitive medicine requires staff approval before completing the sale.")

    # The sale must NOT complete without approval
    assert "Sale Completed" not in body, "Sale completed without the required approval."
    print("Sale is correctly blocked until approval.")

    # Leave a clean state (do not approve in this test)
    driver.find_element(By.XPATH, "//button[text()='Cancel']").click()
    time.sleep(2)

    print("Current URL:", driver.current_url)
    print("PASS: Sensitive Medicine Warning")
except Exception as e:
    print("FAIL: Sensitive Medicine Warning")
    print("Error:", e)
    driver.save_screenshot("21_sensitive_warning_FAIL.png")

In [ ]:
driver.quit()